# A2A: one agent talks to another

MCP connects an agent to tools. A2A connects one agent to another agent, often across organizations.

In this notebook, we will:

- publish an Agent Card for discovery;
- follow a task from submitted to completed;
- pause when the peer needs more input;
- use the official A2A SDK; and
- ask a supplier agent for a restock time.

Before running the SDK section, install the A2A extra once with: pip install -e ".[a2a]".

In [ ]:
import json
import os
import socket
import threading
import urllib.request
from getpass import getpass
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

import litellm
from dotenv import load_dotenv

course_root = Path.cwd()
if not (course_root / "pyproject.toml").exists():
    course_root = course_root.parent
load_dotenv(course_root / ".env")
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")

MODEL = os.getenv("MODEL", "openrouter/deepseek/deepseek-v3.2")
litellm.drop_params = True
litellm.cache = litellm.Cache(type="disk", disk_cache_dir=str(course_root / ".litellm_cache"))

## 1. Publish an Agent Card

An Agent Card says who the peer is, what skill it offers, and where to call it. Clients look for it at a well-known URL.

In [ ]:
CARD_PATH = "/.well-known/agent-card.json"

TOY_CARD = {
    "name": "Learning Supplier",
    "description": "Answers restock questions.",
    "version": "0.1",
    "skills": [{"id": "restock", "name": "Restock lead time"}],
}

TASKS = {}

def next_turn(task_id, text):
    region = next(
        (name for name in ("west", "central", "east") if name in text.lower()),
        None,
    )

    if task_id is None:
        task_id = f"task-{len(TASKS) + 1}"
        TASKS[task_id] = {"region": region}
    elif region:
        TASKS[task_id]["region"] = region

    region = TASKS[task_id]["region"]
    if region is None:
        return {
            "task_id": task_id,
            "state": "input-required",
            "message": "Which region: west, central, or east?",
        }

    return {
        "task_id": task_id,
        "state": "completed",
        "message": f"Restock lead time from {region}: 7 business days.",
    }

The toy server has two endpoints: one for the card and one for task messages.

In [ ]:
class ToyPeer(BaseHTTPRequestHandler):
    def send_json(self, value, status=200):
        body = json.dumps(value).encode()
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        if self.path == CARD_PATH:
            self.send_json(TOY_CARD)
        else:
            self.send_json({}, status=404)

    def do_POST(self):
        size = int(self.headers.get("Content-Length", 0))
        payload = json.loads(self.rfile.read(size) or b"{}")
        self.send_json(next_turn(payload.get("task_id"), payload.get("text", "")))

    def log_message(self, *args):
        pass

with socket.socket() as probe:
    probe.bind(("127.0.0.1", 0))
    toy_port = probe.getsockname()[1]

toy_server = ThreadingHTTPServer(("127.0.0.1", toy_port), ToyPeer)
threading.Thread(target=toy_server.serve_forever, daemon=True).start()
toy_url = f"http://127.0.0.1:{toy_port}"

print(toy_url)

In [ ]:
def get_json(path):
    with urllib.request.urlopen(toy_url + path, timeout=5) as response:
        return json.loads(response.read())

def post_json(value):
    request = urllib.request.Request(
        toy_url + "/tasks",
        data=json.dumps(value).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(request, timeout=5) as response:
        return json.loads(response.read())

card = get_json(CARD_PATH)
print("found:", card["name"], "skill:", card["skills"][0]["id"])

first = post_json({"text": "When can you restock LK-1015?"})
print("first turn:", first)

second = post_json({"task_id": first["task_id"], "text": "west"})
print("second turn:", second)

The peer did not guess the missing region. It moved the task to input-required, kept the task id, and continued after the client replied.

In [ ]:
toy_server.shutdown()
toy_server.server_close()
print("toy peer stopped")

## 2. Use the official A2A SDK

The course includes a supplier agent. The SDK resolves its card, creates a client, and streams task events.

In [ ]:
import httpx
from a2a.client import A2ACardResolver, ClientConfig, ClientFactory
from a2a.helpers import get_stream_response_text, new_text_message
from a2a.types import Role, SendMessageRequest, TaskState
from a2a_servers.supplier_agent import serve_in_background

supplier_server = serve_in_background()
supplier_url = supplier_server.__enter__()
http_client = httpx.AsyncClient(timeout=10)

supplier_card = await A2ACardResolver(
    httpx_client=http_client,
    base_url=supplier_url,
).get_agent_card()

supplier_client = ClientFactory(
    ClientConfig(httpx_client=http_client, streaming=True)
).create(supplier_card)

print("peer:", supplier_card.name)
print("skill:", supplier_card.skills[0].id)

In [ ]:
async def ask_supplier(text, task_id=None, context_id=None):
    message = new_text_message(
        text,
        role=Role.ROLE_USER,
        task_id=task_id,
        context_id=context_id,
    )

    state = None
    answer = ""

    async for event in supplier_client.send_message(SendMessageRequest(message=message)):
        kind = event.WhichOneof("payload")
        if kind == "task":
            state = event.task.status.state
            task_id = event.task.id
            context_id = event.task.context_id
        elif kind == "status_update":
            state = event.status_update.status.state
            task_id = event.status_update.task_id
            context_id = event.status_update.context_id

        answer = get_stream_response_text(event) or answer

    return task_id, context_id, TaskState.Name(state), answer

In [ ]:
task_id, context_id, state, answer = await ask_supplier(
    "What is the restock lead time for LK-1015?"
)
print(state, answer)

task_id, context_id, state, answer = await ask_supplier(
    "west",
    task_id=task_id,
    context_id=context_id,
)
print(state, answer)

## 3. Use the peer in a shop decision

Larkspur knows its stock. The supplier peer knows the restock time. We combine both facts.

In [ ]:
from shoplab import world
from shoplab.llm import llm

product = next(
    item
    for item in world.load_products()
    if item["stock"] <= item["reorder_threshold"]
)

_, _, state, quote = await ask_supplier(
    f"Restock time for {product['sku']} from the west warehouse"
)

recommendation = llm(
    f"Stock is {product['stock']} and reorder threshold is {product['reorder_threshold']}. "
    f"Supplier says: {quote}. Recommend a reorder in one sentence.",
    model=MODEL,
)

print("product:", product["sku"], product["name"])
print("supplier state:", state)
print("recommendation:", recommendation)

## 4. Stop the peer cleanly

Servers and HTTP clients own resources. Close them when the work is done.

In [ ]:
supplier_server.__exit__(None, None, None)
await http_client.aclose()
print("supplier stopped")

## What to remember

1. MCP connects agents to tools; A2A connects peer agents.
2. An Agent Card supports discovery.
3. A task id keeps work connected across turns.
4. input-required pauses instead of guessing missing information.
5. The SDK streams task state and messages.
6. Close peer servers and clients when finished.

## Small practice steps

1. Ask the toy peer for the east region.
2. Print the full supplier Agent Card.
3. Change the stock number in the recommendation prompt.